# 🔴 Implement for MU1 (some codes are common)

## ⚪ 1. Libraries

In [1]:
import pandas as pd
import scipy.io
import numpy as np
from scipy.io import loadmat
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import cdist
from sklearn.metrics import roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

## ⚪ 2. Extract indices from .csv file

In [2]:
# Define the path to your CSV file.
csv_file_path = 'muskori.csv'

In [4]:
# --- Configuration ---
try:
    # --- Step 1: Load the original (raw) dataset using pandas ---
    df_original = pd.read_csv(csv_file_path, header=None)
    print(f"Shape of the original dataset: {df_original.shape} (rows, columns)")
    num_cols = df_original.shape[1]

    # For better readability, let's assign names to the columns.
    column_names = [f"c{i}" for i in range(1, num_cols)] + ["label"]
    df_original.columns = column_names

    # --- Step 2: Apply the condition to define the MU1 subset ---
    condition = (df_original['c1'] >= 28) & (df_original['c1'] <= 60)

    # Create the MU1 subset dataframe by applying the filter
    df_mu1 = df_original[condition]

    # --- Step 3: Extract the indices of the rows that satisfy the condition ---
    mu1_indices = df_mu1.index.to_numpy()

    # --- Step 4" Checking the Number of Outliers in MU1 ---
    mu1_label_counts = df_mu1['label'].value_counts()

    # --- Output and Verification ---
    print(f"Number of samples satisfying the MU1 condition: {len(mu1_indices)}")

    # Display the number of outliers in MU1.
    print(f"Number of outliers in the MU1 subset: {mu1_label_counts}")

except Exception as e:
    print(f"An unexpected error occurred: {e}")

Shape of the original dataset: (3062, 167) (rows, columns)
Number of samples satisfying the MU1 condition: 2963
Number of outliers in the MU1 subset: label
1    2885
2      78
Name: count, dtype: int64


## ⚪ 3. Making X_tem from MU1 indices italicized text

In [5]:
print("Shape of MU1: ", mu1_indices.shape, )
print()
print("index of MU1 :", mu1_indices)
print()
print("type of MU1: ", type(mu1_indices))

Shape of MU1:  (2963,)

index of MU1 : [   0    1    2 ... 3059 3060 3061]

type of MU1:  <class 'numpy.ndarray'>


In [6]:
# Conver mu1_indices to python list
X_tem1 = mu1_indices.tolist()

## ⚪ 4. Checking .mat file

In [7]:
mat = scipy.io.loadmat('musk.mat')
print(mat.keys())
print(pd.DataFrame(mat['trandata']).describe())

# mat['trandata']

dict_keys(['__header__', '__version__', '__globals__', 'trandata'])
               0            1             2            3            4    \
count  3062.000000  3062.000000  3.062000e+03  3062.000000  3062.000000   
mean      0.102680     0.336066  5.044799e-01     0.506212     0.013018   
std       0.068139     0.338443  3.384187e-01     0.324159     0.067784   
min       0.000000     0.000000  1.110223e-16     0.000000     0.000000   
25%       0.056962     0.041522  1.759259e-01     0.218519     0.000000   
50%       0.094937     0.211073  4.629630e-01     0.459259     0.000000   
75%       0.120253     0.536332  8.750000e-01     0.828704     0.002439   
max       1.000000     1.000000  1.000000e+00     1.000000     1.000000   

               5            6            7            8            9    ...  \
count  3062.000000  3062.000000  3062.000000  3062.000000  3062.000000  ...   
mean      0.532476     0.365552     0.537399     0.366496     0.346759  ...   
std       0.239527 

## ⚪ 5. Main code of 3WNCROD

In [8]:
def WNCROD(data, X_tem, lammda):

    n, m = data.shape
    X = np.zeros(n)
    X[X_tem] = 1

    D1 = m / 3
    D2 = m / 2
    D3 = 0.9 * m

    delta = np.zeros(m)
    ID = np.all(data <= 1, axis=0)
    delta[ID] = np.std(data[:, ID], axis=0) / lammda

    Lower = np.zeros((m, n))

    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        Lower_temp = np.min(np.maximum(1 - RM_tem, np.tile(X, (n, 1))), axis=1)
        Lower[col, :Lower_temp.shape[0]] = Lower_temp

    IB = np.tile(X, (m, 1)) - Lower
    NEB = np.min(IB, axis=0)
    NPB = IB - np.tile(NEB, (m, 1))

    n_X = int(sum(X))

    weight = np.zeros((n_X, m))

    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        weight_x = []

        for i in range(n_X):
            temp1 = RM_tem[X_tem[i], :]
            weight_temp = 1 - (np.sqrt((np.sum(np.minimum(temp1, X))) / n_X))
            weight_x.append(weight_temp)

        weight[:len(weight_x), col] = weight_x

    D_tem = np.zeros((n, n))

    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        D_tem += RM_tem

    NOM = m - D_tem
    X_OM = NOM[np.ix_(X_tem, X_tem)]

    NEB_num = np.zeros((n_X, m))
    Lower_num = np.zeros((n_X, m))
    NPB_num = np.zeros((n_X, m))

    for col in range(m):
        temp2 = Lower[col, :]
        temp3 = NPB[col, :]

        for i in range(n_X):
            temp1 = X_OM[i, :]
            NEB_num[i, col] = np.sum(np.minimum(NEB[X_tem], temp1 <= D1))
            NPB_num[i, col] = np.sum(np.minimum(temp3[X_tem], temp1 >= D2))
            Lower_num[i, col] = np.sum(np.minimum(temp2[X_tem], temp1 >= D3))

    MNOF = np.mean(((NEB_num + Lower_num + NPB_num) / n_X) * weight, axis=1)
    return MNOF


## ⚪ 6. Run algorythm for MU1

In [10]:
if __name__ == "__main__":

    # 1) Load data
    mat       = loadmat('musk.mat')
    full_data = mat['trandata']            # shape = (n_samples, 167)

    # 2) Separate features and labels
    X1 = full_data[:, :166]                   # First 166 columns
    y1 =  full_data[:, 166].astype(int)       # Column 167 (index 167)

    # 3) Normalize only the features (columns 1 to 166)
    scaler = MinMaxScaler()
    X1[:, 1:] = scaler.fit_transform(X1[:, 1:])

    # 4) Run the algorithm with the same X_tem and lammda=0.2
    out_scores = WNCROD(X1, X_tem1, lammda=0.2)

    # 5) Extract true labels only for X_tem1
    y_true = y1[X_tem1]

    # 6) Compute metrics
    roc_auc1 = roc_auc_score(y_true, out_scores)
    ap1      = average_precision_score(y_true, out_scores)

    print(f"ROC-AUC: {roc_auc1:.4f}")
    print(f"Average Precision (AP): {ap1:.4f}")

ROC-AUC: 0.9961
Average Precision (AP): 0.8822


In [11]:
if __name__ == "__main__":

    # 1) Load data
    mat       = loadmat('musk.mat')
    full_data = mat['trandata']            # shape = (n_samples, 167)

    # 2) Separate features and labels
    X1 = full_data[:, :166]                   # First 166 columns
    y1 =  full_data[:, 166].astype(int)       # Column 167 (index 167)

    # 3) Normalize only the features (columns 1 to 166)
    scaler = MinMaxScaler()
    X1[:, 1:] = scaler.fit_transform(X1[:, 1:])

    # 4) Run the algorithm with the same X_tem and lammda=0.4
    out_scores = WNCROD(X1, X_tem1, lammda=0.4)

    # 5) Extract true labels only for X_tem1
    y_true = y1[X_tem1]

    # 6) Compute metrics
    roc_auc1 = roc_auc_score(y_true, out_scores)
    ap1      = average_precision_score(y_true, out_scores)

    print(f"ROC-AUC: {roc_auc1:.4f}")
    print(f"Average Precision (AP): {ap1:.4f}")

ROC-AUC: 1.0000
Average Precision (AP): 1.0000


# 🔵 New Approach for MU1

### 1. Main code of new 3WNCROD

In [12]:
def NEW_WNCROD(data: np.ndarray, X_tem: list[int], lammda_list: list[float]) -> np.ndarray:

    n, m = data.shape
    lammda_arr = np.array(lammda_list, dtype=float)
    if lammda_arr.shape[0] != m:
        raise ValueError(f"The length of lammda_list must be equal to the #features : ({m}).")

    X = np.zeros(n, dtype=int)
    X[X_tem] = 1

    D1 = m / 3
    D2 = m / 2
    D3 = 0.9 * m

    # Calculate independent neighborhood radius for each feature
    delta = np.zeros(m, dtype=float)
    ID = np.all(data <= 1, axis=0)
    stds = np.std(data[:, ID], axis=0)
    delta[ID] = stds / lammda_arr[ID]

    Lower = np.zeros((m, n), dtype=float)
    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        Lower_temp = np.min(np.maximum(1 - RM_tem, np.tile(X, (n, 1))), axis=1)
        Lower[col, :Lower_temp.shape[0]] = Lower_temp

    # IB, NEB, NPB
    IB = np.tile(X, (m, 1)) - Lower
    NEB = np.min(IB, axis=0)
    NPB = IB - np.tile(NEB, (m, 1))

    # Weighting
    n_X = int(np.sum(X))
    weight = np.zeros((n_X, m), dtype=float)
    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        wcol = []
        for i in range(n_X):
            temp1 = RM_tem[X_tem[i], :]
            wtemp = 1 - np.sqrt(np.sum(np.minimum(temp1, X)) / n_X)
            wcol.append(wtemp)
        weight[:, col] = wcol

    # D_tem & NOM
    D_tem = np.zeros((n, n), dtype=int)
    for col in range(m):
        RM_tem = cdist(data[:, [col]], data[:, [col]], metric='cityblock') <= delta[col]
        D_tem += RM_tem.astype(int)
    NOM = m - D_tem
    X_OM = NOM[np.ix_(X_tem, X_tem)]

    # counting : NEB_num, Lower_num, NPB_num
    NEB_num = np.zeros((n_X, m), dtype=int)
    Lower_num = np.zeros((n_X, m), dtype=int)
    NPB_num = np.zeros((n_X, m), dtype=int)
    for col in range(m):
        tLower = Lower[col, :]
        tNPB   = NPB[col, :]
        for i in range(n_X):
            row_OM = X_OM[i, :]
            NEB_num[i, col]   = np.sum(np.minimum(NEB[X_tem],    row_OM <= D1))
            NPB_num[i, col]   = np.sum(np.minimum(tNPB[X_tem],   row_OM >= D2))
            Lower_num[i, col] = np.sum(np.minimum(tLower[X_tem], row_OM >= D3))

    # calculate MNOF
    NDF = (NEB_num + Lower_num + NPB_num) / n_X
    MNOF = np.mean(NDF * weight, axis=1)
    return MNOF

### 2. runing algo

In [13]:
# this is for best lammda list for TOP AP
if __name__ == "__main__":

    lammda = [0.4] * 166

    # 1) Load data
    mat       = loadmat('musk.mat')
    full_data = mat['trandata']            # shape = (n_samples, 167)

    # 2) Separate features and labels
    X11 = full_data[:, :166]                   # First 166 columns
    y11 =  full_data[:, 166].astype(int)       # Column 167 (index 166)

    # 3) Normalize only the features (columns 1 to 166)
    scaler = MinMaxScaler()
    X11[:, 1:] = scaler.fit_transform(X11[:, 1:])

    # 4) Run the algorithm with the same X_tem and lammda=0.4 for all
    out_scores11 = NEW_WNCROD(X11, X_tem1, lammda)

    # 5) Extract true labels only for X_tem
    y_true11 = y11[X_tem1]

    # 6) Compute metrics
    roc_auc11 = roc_auc_score(y_true11, out_scores11)
    ap11      = average_precision_score(y_true11, out_scores11)

    print(f"ROC-AUC: {roc_auc11:.4f}")
    print(f"Average Precision (AP): {ap11:.4f}")

ROC-AUC: 1.0000
Average Precision (AP): 1.0000
